# Automotive Data Mapper - MVP: Data Validation Pipeline

**Date:** 2026-08-13  
**Author:** Luis Renteria Lezano  
[LinkedIn](https://www.linkedin.com/in/renteria-luis) | [GitHub](https://github.com/renteria-luis) | [Portfolio](https://luisrenteria.me)

## Executive Summary

* **Goal:** validate the raw automotive service-record feeds before mapping them into the canonical schema. This notebook determines which records the pipeline accepts, which it rejects, and why.
* **Input:** The raw feeds profiled in [`01_sources.ipynb`](01_sources.ipynb), representing three automotive data providers:
  * `shop_a`: CSV
  * `dealer_b`: XML
  * `fleet_c`: JSON
* **Expected input:** 97 records across the three feeds, including 10 records that are known to be impossible to map based on the source characteristics identified during profiling.
* **Data:** [`../data/raw/`](https://github.com/renteria-luis/automotive-data-mapper/tree/main/data/raw)
* **Data dictionary:** [`../docs/DATA_DICTIONARY.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/DATA_DICTIONARY.md)
* **Sample data documentation:** [`../docs/SAMPLE_DATA.md`](https://github.com/renteria-luis/automotive-data-mapper/blob/main/docs/SAMPLE_DATA.md)

This notebook turns the findings from `01_sources.ipynb` into measurable validation rules. The objective is not simply to report that the pipeline rejected 10 records, but to verify that it identified all 10 records that are impossible to map. This makes the validation result measurable: **10 known-invalid records exist, and the pipeline detects 10 of 10.**

In [1]:
import re
from datetime import date, datetime, timezone

import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator

from src.mappings import DEALER_B_MAP, DEALER_B_UNMAPPED, FLEET_C_MAP, SHOP_A_MAP
from src.readers import read_dealer_b, read_fleet_c, read_shop_a

shop_a_raw = read_shop_a()
dealer_b_raw = read_dealer_b()
fleet_c_raw = read_fleet_c()

for name, df in [('shop_a', shop_a_raw), ('dealer_b', dealer_b_raw), ('fleet_c', fleet_c_raw)]:
    print(f'{name:10} {len(df):>3} records')

TOTAL = len(shop_a_raw) + len(dealer_b_raw) + len(fleet_c_raw)
print(f'{"total":10} {TOTAL:>3}')
assert TOTAL == 97

shop_a      42 records
dealer_b    27 records
fleet_c     28 records
total       97


The three reads are now one line each. All the complexity from [`01_sources.ipynb`](01_sources.ipynb) CSV quotes, XML namespace, and JSON envelope; all of it is contained within [`../src/readers.py`](../src/readers.py), so we no longer need to think about it here.

That is what it means for the code to “survive” exploration.

## 1. The canonical schema

A single place defines what an acceptable record is. Everything else refers to it.

### 1.1 The decision: why Pydantic instead of pandas conditions

pandas can validate. `df[df['vin'].str.len() != 17]` returns VINs with the wrong length. It works.

The problem appears when you have two rules at the same time.

In [2]:
demo = pd.DataFrame(
    {
        'record': ['A', 'B', 'C'],
        'vin': ['1FTFW1E50KFA1234', '1G1ZD5ST0LFO04821', '1HGCV1F30LA100234'],
        'description': ['oil change', None, 'brake pads'],
    }
)

bad_len = demo['vin'].str.len() != 17
bad_chars = demo['vin'].str.contains('[IOQ]')
bad_desc = demo['description'].isna()

demo.assign(bad_len=bad_len, bad_chars=bad_chars, bad_desc=bad_desc)

,record,vin,description,bad_len,bad_chars,bad_desc
0,A,1FTFW1E50KFA1234,oil change,True,False,False
1,B,1G1ZD5ST0LFO04821,None,False,True,True
2,C,1HGCV1F30LA100234,brake pads,False,False,False


We would need one column for each condition, and even worse, for record B, what would the rejection code be? It fails two rules. 

With pandas, we would have to manually define which one takes priority, write the order somewhere, and maintain it when rule number eleven is added. The logic would be spread across the masks and the code that combines them.

> **Pydantic reverses the problem:** it validates one record at a time, stops at the first failure, and reports which field failed and why. This is exactly the structure of a reason code.

### 1.2 The ISO 3779 check digit

Of the three VIN checks, two are trivial: 17 characters and no `I`, `O`, or `Q`. The third is the only one that catches a typing error.

Position **9** of the VIN is not vehicle information. It is a number calculated from the other 16 characters. If someone transposes two characters, the calculation no longer matches -> [Official Algorithm Surce](https://www.ecfr.gov/current/title-49/subtitle-B/chapter-V/part-565/subpart-B/section-565.15)
$$\text{Check Digit} = \left( \sum_{i=1}^{17} (v_i \times p_i) \right) \pmod{11}$$
Where $v_i$ is the numerical value of the character at position $i$, y $p_i$ is the multiplier (weight) assigned to that position in the VIN. If the modulo result is 10, the check digit is the letter X.

In [3]:
TRANSLIT = {
    **{c: i + 1 for i, c in enumerate("ABCDEFGH")},
    **{c: i + 1 for i, c in enumerate("JKLMN")},
    "P": 7,
    "R": 9,
    **{c: i + 2 for i, c in enumerate("STUVWXYZ")},
    **{str(d): d for d in range(10)},
}
WEIGHTS = [8, 7, 6, 5, 4, 3, 2, 10, 0, 9, 8, 7, 6, 5, 4, 3, 2]


def vin_check_digit(vin: str) -> str:
    """Position 9 of a VIN, computed from the other sixteen characters (ISO 3779)."""
    total = sum(TRANSLIT[c] * w for c, w in zip(vin, WEIGHTS))
    remainder = total % 11
    return "X" if remainder == 10 else str(remainder)

#### Testing function

[`SAMPLE_DATA.md`](../docs/SAMPLE_DATA.md) says that a `dealer_b` record has two adjacent characters transposed. It has 17 characters and contains no illegal characters, so it passes the first two checks.

In [4]:
good = '1HGCV1F30LA100234'
transposed = good[:2] + good[3] + good[2] + good[4:]

for vin in (good, transposed):
    print(
        f'{vin}  length={len(vin)}  illegal={bool(set(vin) & set("IOQ"))}  '
        f'position 9={vin[8]}  calculated={vin_check_digit(vin)}'
    )

1HGCV1F30LA100234  length=17  illegal=False  position 9=0  calculated=0
1HCGV1F30LA100234  length=17  illegal=False  position 9=0  calculated=7


The check digit test must be run even if the other two checks pass, and they must be run in this order:

**length -> illegal characters -> check digit**

For example, if we ran the check digit test first on a 16-character VIN, the result would be meaningless and could even raise an error.

### 1.3 `VehicleEvent`, and each validator defines its own code

The fields come from [`../docs/DATA_DICTIONARY.md`](../docs/DATA_DICTIONARY.md). The decision we need to make here is **how the reason code gets from the validator to the rejection table**.

The solution: each validator puts its code at the beginning of the error message. Adding a new rule means adding a validator with its code in one place, without having to maintain a separate translation table.

Two details of the model:

* `extra='forbid'` makes an unknown field an error instead of letting it pass unnoticed. This is the opposite of silent behavior.
* `mode='before'` makes the validator run **before** Pydantic attempts to convert the value's type. Without it, an invalid date would fail with Pydantic's generic error message instead of the `E005` code.

In [5]:
class VehicleEvent(BaseModel):
    """One service event on one vehicle, in the canonical shape."""

    # forbig: reject | ignore: ignores | allow: keeps
    model_config = ConfigDict(extra="forbid")

    source_id: str
    source_record_id: str
    vin: str
    # this will remain None within this version, E003 rejects => False not possible
    vin_valid: bool | None = None  
    event_date: date
    odometer_km: int | None = None
    odometer_source_unit: str | None = None
    raw_description: str
    normalized_description: str
    provider_name: str | None = None
    provider_city: str | None = None
    provider_province: str | None = None
    ingested_at: datetime  # pydantic not only validates but also converts

    @field_validator("vin", mode="before")
    @classmethod
    def check_vin(cls, v):
        if v is None or str(v).strip() == "":  # not empty
            raise ValueError("E004: vin is empty")
        v = str(v).strip().upper()             # convert to upper+str
        if len(v) != 17:                       # 17 chars
            raise ValueError(f"E001: vin has {len(v)} characters, expected 17")
        illegal = sorted(set(v) & set("IOQ"))
        if illegal:                           # not I, O, Q chars
            raise ValueError(f"E002: vin contains {illegal}, which are not used in VINs")
        expected = vin_check_digit(v)
        if v[8] != expected:                  # check digit
            raise ValueError(f"E003: check digit is {v[8]}, expected {expected}")
        return v

    @field_validator("event_date", mode="before")
    @classmethod
    def check_date(cls, v):
        if v is None or str(v).strip() == "":
            raise ValueError("E004: event_date is empty")
        try:
            parsed = date.fromisoformat(str(v))
        except ValueError:
            raise ValueError(f"E005: {v!r} is not a real calendar date")
        if parsed > date.today():
            raise ValueError(f"E006: {parsed.isoformat()} is in the future")
        return parsed

    @field_validator("source_record_id", "raw_description", mode="before")
    @classmethod
    def check_not_empty(cls, v, info):  # need info to know which field is validating
        if v is None or str(v).strip() == "":
            raise ValueError(f"E004: {info.field_name} is empty")
        return str(v).strip()

    @field_validator("odometer_km")  # mode: after, we need it to be converted to int first
    @classmethod
    def check_odometer(cls, v):
        if v is not None and v < 0:
            raise ValueError(f"E008: odometer is {v}, which cannot be negative")
        return v

### 1.4 `RejectedRecord`

A record that fails **is not an error; it is data**. That is the decision in DD-006, and it defines the structure of this table.

`raw_record` is the field that makes the difference. Without it, fixing a rejected record means manually searching for it in the original file. With it, the record travels together with its reason and can be corrected and reprocessed.

In [6]:
class RejectedRecord(BaseModel):
    """One record the pipeline could not turn into a VehicleEvent."""

    source_id: str
    source_record_id: str | None = None
    reason_code: str
    detail: str
    raw_record: dict
    ingested_at: datetime

`source_record_id` must be None for `E010`: when an unmapped column is detected, because this error refers to the entire file and not to a specific record. `None` is also valid for a record that is so broken that its ID cannot be read. In that case, `raw_record` is what remains available for investigation, that is why this field is mandatory.

## 2. Mapping & Normalization

Here we apply what [`01_sources.ipynb`](01_sources.ipynb) defined. The mapping restructures and transforms; it does not judge if something is valid or not. All validation decisions belong in section 3.

This separation is deliberate: if mapping also rejected records, there would be two different places producing reason codes, and neither would have the complete list.

In [7]:
# This function thoroughly cleans strings and returns None if the value is null or NaN
# will be used in all mappings for all strings
def clean(value) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return str(value).strip() or None

### 2.1 Shared transformations among feeds
3 functions that the 3 of the feeds use. Rules come frrom profiling stage [`01_sources.ipynb`](01_sources.ipynb)

In [8]:
def to_km(value, unit: str | None) -> int | None:
    text = clean(value)
    if text is None:
        return None
    text = text.replace(',', '')
    try:
        number = int(float(text))  # int('1200.0') would not pass
    except ValueError:
        return None  # cannot convert? => None
    if number == 0:
        return None
    return round(number * 1.609344) if unit == 'mi' else number


def normalize(text) -> str | None:
    text = clean(text)
    if text is None:
        return None
    return re.sub(r'\s+', ' ', text).lower() or None


PROVIDER_NAMES = {
    'riverside auto service': 'Riverside Auto Service',
    'riverside auto service ltd': 'Riverside Auto Service',
    'forest city motors': 'Forest City Motors',
    'forest city motors ltd': 'Forest City Motors',
}


def standardize_provider(name) -> str | None:
    text = clean(name)
    if text is None:
        return None
    return PROVIDER_NAMES.get(normalize(text), text)

**Two decisions are documented in `to_km`.**

A zero odometer reading is treated as unknown, not as 0 km. A vehicle with a service history is unlikely to have zero mileage, so that zero is almost certainly a field the shop left blank and the system filled with a default value.

And if a value cannot be converted, `None` is returned instead of raising an error. A missing odometer makes the record **incomplete**, not **invalid**, and [`SAMPLE_DATA.md`](../docs/SAMPLE_DATA.md) states this explicitly: rejecting it would be just as incorrect as accepting a broken VIN.

### 2.2 One mapper per feed

Each mapper applies its dictionary and the transformations justified by the profiling. Dates are normalized to ISO text here, but if they cannot be parsed, **the original value is passed through unchanged**: mapping does not make decisions; it only prepares the data.

#### `map_shop_a`

In [9]:
# computed once, and the same timestamp is used for all of the mapped records
INGESTED_AT = datetime.now(timezone.utc)

def map_shop_a(row) -> dict:
    source_record_id = clean(row['RO_INVOICE_NUMBER'])
    vin = clean(row['VIN'])
    unit = normalize(row['ODOMETER_MEASURE'])
    odometer_km = to_km(row['MILEAGE'], unit)
    raw_description = clean(row['SERVICE_DESCRIPTION'])
    normalized_description = normalize(raw_description)
    provider_name = standardize_provider(row['LOCATION_NAME'])
    provider_city = clean(row['CITY'])
    provider_province = (clean(row['STATE']) or '').upper() or None

    try:
        event_date = datetime.strptime(str(row['RO_OPEN_DATE']).strip(), '%m/%d/%Y').date().isoformat()
    except ValueError:
        event_date = clean(row['RO_OPEN_DATE'])

    return {
        'source_id': 'shop_a',
        'source_record_id': source_record_id,
        'vin': vin,
        'event_date': event_date,
        'odometer_km': odometer_km,
        'odometer_source_unit': unit,
        'raw_description': raw_description,
        'normalized_description': normalized_description,
        'provider_name': provider_name,
        'provider_city': provider_city,
        'provider_province': provider_province,
        'ingested_at': INGESTED_AT,
    }

#### `map_dealer_b`

In [10]:
UNIT_BY_CODE = {'KMT': 'km', 'SMI': 'mi'}

def map_dealer_b(row) -> dict:
    """
    considerations:
    - event id not exist in the file, it is built from the order and the job id. 
    - the correction wins over the customer concern.
    """
    source_record_id = f"{clean(row['DocumentID'])}-{clean(row['JobID'])}"
    vin = clean(row['VehicleID'])
    unit = UNIT_BY_CODE.get(row['unitCode'])
    odometer_km = to_km(row['InDistanceMeasure'], unit)
    raw_description = clean(row['CorrectionDescription']) or clean(row['CustomerConcernDescription'])
    normalized_description = normalize(raw_description)
    provider_name = standardize_provider(row['OrganizationName'])
    provider_city = clean(row['CityName'])
    provider_province = (clean(row['StateOrProvinceCountrySubDivisionID']) or '').upper() or None

    try:
        event_date = datetime.fromisoformat(str(row['RepairOrderOpenedDate'])).date().isoformat()
    except (ValueError, TypeError):
        event_date = clean(row['RepairOrderOpenedDate'])

    return {
        'source_id': 'dealer_b',
        'source_record_id': source_record_id,
        'vin': vin,
        'event_date': event_date,
        'odometer_km': odometer_km,
        'odometer_source_unit': unit,
        'raw_description': raw_description,
        'normalized_description': normalized_description,
        'provider_name': provider_name,
        'provider_city': provider_city,
        'provider_province': provider_province,
        'ingested_at': INGESTED_AT,
    }

#### `map_fleet_c`

In [11]:
def map_fleet_c(row) -> dict:
    source_record_id = clean(row['work_order_id'])
    vin = clean(row['vin'])
    unit = normalize(row['odometer.unit'])
    odometer_km = to_km(row['odometer.value'], unit)
    raw_description = clean(row['description'])
    normalized_description = normalize(raw_description)
    provider_name = standardize_provider(row['vendor.name'])
    provider_city = clean(row['vendor.city'])
    provider_province = (clean(row['vendor.region']) or '').upper() or None

    try:
        stamp = str(row['service_date']).replace('Z', '+00:00')
        event_date = datetime.fromisoformat(stamp).date().isoformat()
    except ValueError:
        event_date = str(row['service_date'])[:10]

    return {
        'source_id': 'fleet_c',
        'source_record_id': source_record_id,
        'vin': vin,
        'event_date': event_date,
        'odometer_km': odometer_km,
        'odometer_source_unit': unit,
        'raw_description': raw_description,
        'normalized_description': normalized_description,
        'provider_name': provider_name,
        'provider_city': provider_city,
        'provider_province': provider_province,
        'ingested_at': INGESTED_AT,
    }

#### Testing `to_km`

In [12]:
samples = [
    ('shop_a  "21,000" KM ', to_km('21,000', 'km')),
    ('shop_a  "30733"  MI ', to_km('30733', 'mi')),
    ('shop_a  empty       ', to_km(None, 'km')),
    ('shop_a  zero        ', to_km('0', 'km')),
    ('dealer_b "88000" SMI', to_km('88000', 'mi')),
    ('fleet_c  "44,032" km', to_km('44,032', 'km')),
    ('fleet_c  null       ', to_km(None, 'km')),
]

for label, value in samples:
    print(f'{label} -> {value}')

shop_a  "21,000" KM  -> 21000
shop_a  "30733"  MI  -> 49460
shop_a  empty        -> None
shop_a  zero         -> None
dealer_b "88000" SMI -> 141622
fleet_c  "44,032" km -> 44032
fleet_c  null        -> None


Here there is a limitation and an opportunity for improvement: the transformer cannot distinguish between `'0'`, an empty field, and `'twenty thousand'`. It treats all three scenarios as `None`.

Ideally, we would distinguish between zero and blank values, while preserving **unparseable values** such as `'twenty thousand'`, so someone can audit and correct them if necessary. The odometer is not a required field, so a missing value should not cause the record to be rejected; however, adding this feature would still improve data quality and auditability.
## 3. Validation
### 3.1 what slips through without validation
Four records with real defects from the source files. Without Pydantic, all four would get through.

In [13]:
suspects = [
    {'label': '16-character VIN', 'vin': '1FTFW1E50KFA1234', 'event_date': '2019-10-10'},
    {'label': 'VIN with O', 'vin': '1G1ZD5ST0LFO04821', 'event_date': '2022-12-22'},
    {'label': 'Transposed VIN', 'vin': transposed, 'event_date': '2023-07-26'},
    {'label': 'February 30 + Future', 'vin': good, 'event_date': '2030-02-30'},
]

for s in suspects:
    accepted = s['vin'] is not None and s['event_date'] is not None
    print(f"{s['label']:20} : Does a pipeline without validation accept it? -> {'yes' if accepted else 'no'}")

16-character VIN     : Does a pipeline without validation accept it? -> yes
VIN with O           : Does a pipeline without validation accept it? -> yes
Transposed VIN       : Does a pipeline without validation accept it? -> yes
February 30 + Future : Does a pipeline without validation accept it? -> yes


> Everything that is not `None` will be accepted, that is because we made Pydantic responsible for the validation

### 3.2 What Pydantic returns when fails

In [14]:
try:
    VehicleEvent(
        source_id='shop_a',
        source_record_id='184227',
        vin='1FTFW1E50KFA1234',
        event_date='2019-10-10',
        raw_description='oil change',
        normalized_description='oil change',
        ingested_at=INGESTED_AT,
    )
except ValidationError as error:
    for e in error.errors():
        print('field:', e['loc'])
        print('type:', e['type'])
        print('message:', e['msg'])

field: ('vin',)
type: value_error
message: Value error, E001: vin has 16 characters, expected 17


In [15]:
def reason_code(error: dict) -> str:
    """The code the validator declared, or E004 for Pydantic's own missing-field errors."""
    found = re.search(r"E0\d{2}", error.get("msg", ""))
    if found:
        return found.group(0)
    return {"missing": "E004"}.get(error["type"], "E004")

> A record can fail more than one rule, and Pydantic reports all of them through `errors()`. However, the rejection table stores only one code. This involves a deliberate loss of information: a rejected record leaves the pipeline regardless of how many problems it has, so it will be sent for review anyway. However, whoever investigates the record needs to know what to fix first, which is why the validators have a specific order of precedence.

### 3.3 Exact duplicate within the feed

There are two levels of deduplication:
- the exact same row appearing twice in the same file: detected during ingestion
- the same event written differently or arriving from two sources: requires everything to be normalized, belongs in section 5.

## 4. Run the pipeline

A single loop handles all three feeds. The structure is always the same: map, check for exact duplicates, validate, and send the result to one of the two tables.

Nothing is silently discarded: every record read ends up in either `events` or `rejected`.

In [16]:
FEEDS = [
    ('shop_a', shop_a_raw, map_shop_a),
    ('dealer_b', dealer_b_raw, map_dealer_b),
    ('fleet_c', fleet_c_raw, map_fleet_c),
]

events: list[VehicleEvent] = []
rejected: list[RejectedRecord] = []

for source_id, df, mapper in FEEDS:
    seen: set = set()

    for _, row in df.iterrows():
        # for each feed -> map the row into the canonical shape
        payload = mapper(row)
        raw = row.to_dict()

        # before validating, check whether seen an identical row from that feed already
        fingerprint = tuple(sorted((k, str(v)) for k, v in raw.items()))
        if fingerprint in seen:
            rejected.append(
                RejectedRecord(
                    source_id=source_id,
                    source_record_id=payload['source_record_id'],
                    reason_code='E009',
                    detail='identical row already read from this feed',
                    raw_record=raw,
                    ingested_at=INGESTED_AT,
                )
            )
            continue
        seen.add(fingerprint)

        # pydantic handles the payload, whatever it validates becomes an event
        # if fails take first error pulling reason code and write with its row
        try:
            events.append(VehicleEvent(**payload))
        except ValidationError as error:
            first = error.errors()[0]
            rejected.append(
                RejectedRecord(
                    source_id=source_id,
                    source_record_id=payload['source_record_id'],
                    reason_code=reason_code(first),
                    detail=first['msg'].replace('Value error, ', ''),
                    raw_record=raw,
                    ingested_at=INGESTED_AT,
                )
            )

print(f'read: {TOTAL}')
print(f'mapped: {len(events)}')
print(f'rejected: {len(rejected)}')

# accepted plus rejected has to equal what we read
# if not, a record disappeared without anyone reporting it
assert len(events) + len(rejected) == TOTAL, 'a record was lost along the way'

read: 97
mapped: 87
rejected: 10


> **This last `assert` is the most important check in the notebook:**
> 
>**Accepted plus rejected records must equal the total number of records read. If they do not match, a record disappeared without being reported, which is exactly the kind of failure this project exists to prevent.**

### 4.1 The exception queue

In [17]:
rejected_df = pd.DataFrame([r.model_dump() for r in rejected])
rejected_df[['source_id', 'source_record_id', 'reason_code', 'detail']].sort_values(
    ['reason_code', 'source_id']
).reset_index(drop=True)

,source_id,source_record_id,reason_code,detail
0,shop_a,184227,E001,"E001: vin has 16 characters, expected 17"
1,shop_a,184209,E002,"E002: vin contains ['O'], which are not used i..."
2,dealer_b,RO-100494-1,E003,"E003: check digit is 0, expected 1"
3,dealer_b,RO-100557-1,E004,E004: raw_description is empty
4,fleet_c,WO-2026-4180,E004,E004: raw_description is empty
5,shop_a,184269,E004,E004: raw_description is empty
6,fleet_c,WO-2026-4145,E005,E005: '2026-02-30' is not a real calendar date
7,shop_a,184278,E005,E005: '00/00/0000' is not a real calendar date
8,dealer_b,RO-100536-1,E006,E006: 2027-11-10 is in the future
9,shop_a,184302,E009,identical row already read from this feed


### 4.2 Compare against the pre-written list

[`docs/SAMPLE_DATA.md`](../docs/SAMPLE_DATA.md) specifies which 10 records are impossible to map and which code each one should receive. Here we compare the actual results against that expected list. 

In [18]:
EXPECTED = {
    ('shop_a', '184227'): 'E001',
    ('shop_a', '184209'): 'E002',
    ('dealer_b', 'RO-100494-1'): 'E003',
    ('shop_a', '184269'): 'E004',
    ('dealer_b', 'RO-100557-1'): 'E004',
    ('fleet_c', 'WO-2026-4180'): 'E004',
    ('shop_a', '184278'): 'E005',
    ('fleet_c', 'WO-2026-4145'): 'E005',
    ('dealer_b', 'RO-100536-1'): 'E006',
    ('shop_a', '184302'): 'E009',
}

found = {
    (r['source_id'], r['source_record_id']): r['reason_code']
    for _, r in rejected_df.iterrows()
}

print(f'{"record":28} {"expected":10} {"found":10}')
for key, expected in EXPECTED.items():
    got = found.get(key, 'not detected')
    flag = '' if got == expected else '   <-- review'
    print(f'{key[0] + " " + key[1]:28} {expected:10} {got:10}{flag}')

print()
print(f'detected {sum(1 for k, v in EXPECTED.items() if found.get(k) == v)} of {len(EXPECTED)}')
print(f'false positives: {len(found) - len(EXPECTED & found.keys())}')

record                       expected   found     
shop_a 184227                E001       E001      
shop_a 184209                E002       E002      
dealer_b RO-100494-1         E003       E003      
shop_a 184269                E004       E004      
dealer_b RO-100557-1         E004       E004      
fleet_c WO-2026-4180         E004       E004      
shop_a 184278                E005       E005      
fleet_c WO-2026-4145         E005       E005      
dealer_b RO-100536-1         E006       E006      
shop_a 184302                E009       E009      

detected 10 of 10
false positives: 0


- This table shows the difference between *data validation* and *data quality assurance*.
- Validation is the set of rules `E001` through `E006`. Quality assurance is this: a known list written before the code, an explicit comparison, and a number that can be defended.
- Without it, “the pipeline rejected 10 records” is a claim about the pipeline. With it, “it found 10 out of the 10 expected records” is a measurement.
